# Table 2 Replication
This notebook replicates the Global and Japan sections of Table 2 in Fama and French (2012).

Panel A reports the mean and standard deviation of monthly excess returns for 25 portfolios formed on size and book-to-market.

Panel B reports the same statistics for 25 portfolios formed on size and previous returns.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_DIR = Path("cleaned_data")

In [3]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [4]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [5]:
developed_size_bm = pd.read_csv(
    DATA_DIR / "developed_25_size_bm.csv",
    parse_dates=["date"]
)

In [6]:
japan_size_bm = pd.read_csv(
    DATA_DIR / "japan_25_size_bm.csv",
    parse_dates=["date"]
)

In [7]:
developed_size_momentum = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv",
    parse_dates=["date"]
)

In [8]:
japan_size_momentum = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv",
    parse_dates=["date"]
)

In [9]:
print("Developed size and B/M:", len(developed_size_bm))
print("Japan size and B/M:", len(japan_size_bm))
print("Developed size and momentum:", len(developed_size_momentum))
print("Japan size and momentum:", len(japan_size_momentum))

Developed size and B/M: 245
Japan size and B/M: 245
Developed size and momentum: 245
Japan size and momentum: 245


## Matching portfolio returns with the risk-free rate

Before calculating excess returns, match the portfolio data with the factor data using `date`.

Use the risk-free rate from the corresponding market:

- Use Developed `RF` for both Developed portfolio datasets.
- Use Japanese `RF` for both Japanese portfolio datasets.

For each month, subtract the same market-specific `RF` from all 25 portfolio returns.

Subtract only the `RF` column. Do not subtract `Mkt-RF`.

The portfolio returns and `RF` are already expressed as percentages. Do not divide them by 100 or annualize them.

## Task 1: Calculate Excess Returns

Table 2 reports excess portfolio returns.

For every month, calculate:

Portfolio excess return = portfolio return − risk-free return

The risk-free rate, `RF`, is available in the factor datasets loaded above.

Calculate excess returns for these four combinations:

1. Developed size and book-to-market portfolios
2. Japan size and book-to-market portfolios
3. Developed size and momentum portfolios
4. Japan size and momentum portfolios

Keep the date and all 25 excess-return columns.

In [10]:
def excess_returns(portfolios, factors):
    """Match months on date and subtract the market's RF from all 25 portfolio returns."""
    merged = portfolios.merge(factors[["date", "RF"]], on="date", validate="one_to_one")
    portfolio_columns = portfolios.columns.drop("date")

    excess = merged[portfolio_columns].sub(merged["RF"], axis=0)
    excess.insert(0, "date", merged["date"])
    return excess


developed_size_bm_excess = excess_returns(developed_size_bm, developed_factors)
japan_size_bm_excess = excess_returns(japan_size_bm, japan_factors)
developed_size_momentum_excess = excess_returns(developed_size_momentum, developed_factors)
japan_size_momentum_excess = excess_returns(japan_size_momentum, japan_factors)

excess_datasets = {
    "Developed size and B/M": developed_size_bm_excess,
    "Japan size and B/M": japan_size_bm_excess,
    "Developed size and momentum": developed_size_momentum_excess,
    "Japan size and momentum": japan_size_momentum_excess,
}

for name, excess in excess_datasets.items():
    n_portfolios = len(excess.columns) - 1
    print(f"{name}: {len(excess)} months, {n_portfolios} portfolios")

Developed size and B/M: 245 months, 25 portfolios
Japan size and B/M: 245 months, 25 portfolios
Developed size and momentum: 245 months, 25 portfolios
Japan size and momentum: 245 months, 25 portfolios


## Constructing the Table 2 results

Calculate the statistics separately for every portfolio.

For each portfolio:

1. Use its 245 monthly excess returns.
2. Calculate one arithmetic mean.
3. Calculate one sample standard deviation.

This produces 25 means and 25 standard deviations for each dataset.

Do not calculate the mean across the 25 portfolios. The averaging is performed across the 245 months for each portfolio separately.

Arrange the results in the same order as the portfolio columns in the data:

- The first five portfolio columns form the `Small` row.
- The next five form size row `2`.
- The next five form size row `3`.
- The next five form size row `4`.
- The final five form the `Big` row.

## Task 2: Report the Portfolio Statistics

For each of the four combinations, calculate:

1. Mean monthly excess return
2. Standard deviation of monthly excess return

Arrange each result as a 5 × 5 table (refer to the paper to understand the structure).

For the size and book-to-market portfolios:

- Rows should move according to size from Small to Big.
- Columns should move from Low book-to-market to High book-to-market.

For the size and momentum portfolios:

- Rows should move according to size from Small to Big.
- Columns should move according to the momentum factor from Losers to Winners.

Round the results to two decimal places.

In [11]:
from IPython.display import display

SIZE_LABELS = ["Small", "2", "3", "4", "Big"]
BM_LABELS = ["Low", "2", "3", "4", "High"]
MOMENTUM_LABELS = ["Losers", "2", "3", "4", "Winners"]


def to_grid(values, column_labels):
    """Arrange 25 portfolio values as 5x5: first five columns are Small, last five are Big."""
    return pd.DataFrame(
        values.to_numpy().reshape(5, 5),
        index=SIZE_LABELS,
        columns=column_labels,
    )


def portfolio_grids(excess, column_labels):
    """Mean and sample standard deviation of each portfolio across its 245 months."""
    returns = excess.drop(columns="date")
    mean_grid = to_grid(returns.mean(), column_labels).round(2)
    std_grid = to_grid(returns.std(), column_labels).round(2)
    return mean_grid, std_grid


grid_inputs = {
    "Panel A, Global (Developed): size and B/M": (developed_size_bm_excess, BM_LABELS),
    "Panel A, Japan: size and B/M": (japan_size_bm_excess, BM_LABELS),
    "Panel B, Global (Developed): size and momentum": (developed_size_momentum_excess, MOMENTUM_LABELS),
    "Panel B, Japan: size and momentum": (japan_size_momentum_excess, MOMENTUM_LABELS),
}

table2_grids = {}

for title, (excess, column_labels) in grid_inputs.items():
    mean_grid, std_grid = portfolio_grids(excess, column_labels)
    table2_grids[title] = {"Mean": mean_grid, "Standard deviation": std_grid}

    print(title)
    display(pd.concat({"Mean": mean_grid, "Standard deviation": std_grid}, axis=1))

Panel A, Global (Developed): size and B/M


Mean                         Standard deviation                        
        Low     2     3     4  High                Low     2     3     4  High
Small  0.05  0.48  0.77  0.78  1.12               6.07  5.59  5.31  4.60  4.38
2      0.08  0.42  0.53  0.67  0.80               6.01  5.32  4.65  4.39  4.49
3      0.19  0.39  0.52  0.58  0.76               5.88  5.26  4.69  4.46  4.60
4      0.40  0.43  0.47  0.62  0.68               5.78  4.64  4.54  4.41  4.73
Big    0.28  0.37  0.48  0.52  0.54               4.65  4.30  4.45  4.48  5.25

Panel A, Japan: size and B/M


Mean                         Standard deviation                        
        Low     2     3     4  High                Low     2     3     4  High
Small -0.15 -0.05  0.05  0.10  0.26               9.47  7.95  7.76  7.20  7.31
2     -0.39 -0.39 -0.14  0.03  0.03               8.46  7.74  7.44  7.13  7.23
3     -0.51 -0.40 -0.27 -0.12  0.13               8.16  7.07  6.68  6.45  6.99
4     -0.54 -0.22 -0.16  0.01  0.05               7.51  6.50  6.04  6.05  6.84
Big   -0.30 -0.10 -0.10  0.19  0.33               6.94  5.93  6.18  6.00  7.37

Panel B, Global (Developed): size and momentum


Mean                           Standard deviation                    \
      Losers     2     3     4 Winners             Losers     2     3     4   
Small   0.15  0.63  0.79  1.12    1.56               6.41  4.35  3.91  4.11   
2       0.14  0.46  0.55  0.80    1.12               6.73  4.67  4.19  4.20   
3       0.28  0.46  0.54  0.57    0.87               6.73  4.87  4.28  4.22   
4       0.25  0.41  0.54  0.54    0.87               6.70  4.82  4.21  4.16   
Big     0.11  0.31  0.39  0.55    0.63               6.31  4.64  4.09  4.16   

               
      Winners  
Small    5.43  
2        5.59  
3        5.54  
4        5.40  
Big      5.31

Panel B, Japan: size and momentum


Mean                           Standard deviation                    \
      Losers     2     3     4 Winners             Losers     2     3     4   
Small   0.15  0.30  0.13  0.28   -0.01               8.84  7.18  6.62  6.58   
2      -0.14 -0.05  0.01 -0.03   -0.10               8.72  7.07  6.64  6.70   
3      -0.21 -0.22 -0.13 -0.03   -0.05               8.09  6.87  6.07  6.19   
4      -0.11 -0.11 -0.14 -0.17   -0.01               8.02  6.54  6.07  5.91   
Big    -0.08 -0.30 -0.28 -0.10   -0.07               8.36  6.54  6.15  5.88   

               
      Winners  
Small    8.02  
2        7.45  
3        6.98  
4        6.78  
Big      6.86

## Task 3: Compare and Interpret the Results

Compare your size and book-to-market results with the Global and Japan sections of Table 2, Panel A.

Compare your size and momentum results with the Global and Japan sections of Table 2, Panel B.

Identify the main return patterns and differences that you would report. Interpret what the results show about value and momentum across company sizes and across the two markets.

Your values may differ slightly because the Kenneth French database has been updated since the paper was published.

Note your findings in your report.